# 21 · Advanced Analytics and Visualisation

EarthNets catalog analysis: volume trends, correlation matrices, clustering, radar charts.

## Contents
1. Catalog analysis report
2. Volume trend
3. Domain distribution
4. Correlation matrix
5. Radar chart comparison
6. Dataset clustering (t-SNE)

## 1 · Full Catalog Analysis

In [ ]:
from pygeovision.datasets.analysis import DatasetAnalyzer
from pygeovision.datasets.registry import dataset_registry

analyzer = DatasetAnalyzer(dataset_registry)
report = analyzer.full_report()
s = report['catalog_summary']
print('EarthNets Dataset Catalog Analysis')
print('=' * 50)
print(f'  Total datasets:  {s["total_datasets"]}')
print(f'  Total volume:    {s["total_volume_tb"]} TB')
print(f'  Year range:      {s["year_range"][0]} - {s["year_range"][1]}')
print(f'  Domains:         {len(s["domains"])}')
print(f'  Tasks:           {len(s["tasks"])}')
print()
for task in ['segmentation','detection','classification','change_detection']:
    names = report.get(f'top_{task}', [])
    print(f'  Top-{task}: {names[:3]}')

EarthNets Dataset Catalog Analysis
  Total datasets:  220
  Total volume:    20.6 TB
  Year range:      1992 - 2024
  Domains:         24
  Tasks:           14

  Top-segmentation: ['TUM-Urban', 'SensatUrban', 'DublinCity']
  Top-detection: ['TreeSense', 'SARdet-100K', 'CrowdAI Mapping']
  Top-classification: ['Skysense', 'GFM', 'RingMo']
  Top-change_detection: ['DynamicEarthNet', 'LEVIR-CD', 'WHU-CD']


## 2 · Volume Growth Trend

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

trend = analyzer.volume_trend(show=False)
years = sorted(trend.keys())
vols  = [trend[y] for y in years]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].fill_between(years, vols, alpha=0.3, color='steelblue')
axes[0].plot(years, vols, 'o-', color='steelblue', linewidth=2, markersize=5)
axes[0].set_xlabel('Year'); axes[0].set_ylabel('Cumulative Volume (TB)')
axes[0].set_title('EarthNets: Cumulative Dataset Volume', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

yoy = {y: trend[y] - trend.get(y-1, 0) for y in sorted(trend.keys())[1:]}
axes[1].bar(list(yoy.keys()), list(yoy.values()), color='steelblue', edgecolor='white')
axes[1].set_xlabel('Year'); axes[1].set_ylabel('New Volume Added (TB)')
axes[1].set_title('Annual Dataset Volume Additions', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
import os; os.makedirs('./results/analytics', exist_ok=True)
plt.savefig('./results/analytics/volume_trend.png', dpi=150)
plt.show()
print(f'Cumulative volume 2024: {vols[-1]:.1f} TB')

## 3 · Domain and Modality Distribution

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

domains = analyzer.domain_distribution(show=False)
sorted_d = sorted(domains.items(), key=lambda x: -x[1])[:12]
axes[0].barh([d[0] for d in sorted_d[::-1]], [d[1] for d in sorted_d[::-1]], color='steelblue', edgecolor='white')
axes[0].set_xlabel('Number of Datasets'); axes[0].set_title('Datasets by Domain (Top-12)', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

mods = analyzer.modality_distribution(show=False)
sorted_m = sorted(mods.items(), key=lambda x: -x[1])
axes[1].pie([m[1] for m in sorted_m], labels=[m[0] for m in sorted_m],
             autopct='%1.1f%%', startangle=90)
axes[1].set_title('Datasets by Modality', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('./results/analytics/distributions.png', dpi=150)
plt.show()

## 4 · Correlation Matrix

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

key = ['EuroSAT','BigEarthNet','LoveDA','LEVIR-CD','DOTA','SAR-Ship','AgriFieldNet']
matrix = analyzer.correlation_matrix(names=key, show=False)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(matrix, cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='EarthNets Similarity')
ax.set_xticks(range(len(key))); ax.set_yticks(range(len(key)))
ax.set_xticklabels(key, rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(key, fontsize=10)
ax.set_title('EarthNets Dataset Correlation Matrix', fontsize=14, fontweight='bold')
for i in range(len(key)):
    for j in range(len(key)):
        ax.text(j, i, f'{matrix[i][j]:.2f}', ha='center', va='center',
                fontsize=8, color='white' if matrix[i][j] > 0.6 else 'black')
plt.tight_layout()
plt.savefig('./results/analytics/correlation_matrix.png', dpi=150)
plt.show()
print('High correlations (>0.7):')
for i, a in enumerate(key):
    for j, b in enumerate(key):
        if i < j and matrix[i][j] > 0.7:
            print(f'  {a} ~ {b}: {matrix[i][j]:.2f}')

## 5 · Radar Chart

In [ ]:
datasets_to_compare = ['EuroSAT', 'BigEarthNet', 'DOTA', 'LEVIR-CD', 'SensatUrban']
fig = analyzer.radar_chart(datasets_to_compare, save_path='./results/analytics/radar.png', show=False)
if fig:
    import matplotlib.pyplot as plt
    plt.show()
    print(f'Radar chart: {datasets_to_compare}')
    print('Dimensions: Resolution | Scale | Diversity | Recency | Volume | Access')
else:
    print('Radar chart requires matplotlib: pip install matplotlib')

## 6 · Dataset Clustering

In [ ]:
print('Clustering 220+ datasets in 2D with t-SNE...')
embedding = analyzer.cluster(
    method='tsne',
    save_path='./results/analytics/cluster_tsne.png',
)
if embedding is not None:
    print(f'Embedding: {embedding.shape}')
    print('Clusters visible: urban | agriculture | SAR | time-series')
    print('Plot: ./results/analytics/cluster_tsne.png')
else:
    print('Install: pip install scikit-learn')
    print('Or UMAP: pip install umap-learn')
    print()
    print('Usage:')
    print("  embedding = analyzer.cluster(method='tsne', save_path='cluster.png')")
    print("  embedding = analyzer.cluster(method='umap', save_path='cluster_umap.png')")